# 13 — Naive baseline

A zero-information reference for the model notebooks. For each of the 7 event
fields, always predict the same constant, estimated from the **training set
only**:

- **mode** for the categorical fields (`is_home`, `event`, `result`) — scored by accuracy
- **median** for the numeric fields (`start_x`, `start_y`, `time_elapsed`, `duration`) — scored by MAE (the median is the MAE-optimal constant)

It also reports the **cross-entropy of each field's training marginal**, which
sums to a number directly comparable to the disjoint-heads model's summed
per-field cross-entropy loss.

Val split. (`14_full_event_evaluation.ipynb` runs the same baseline alongside
v5 and is the notebook that gets pointed at the test split.)

Rows are grouped exactly like the models see them: the rows predicting field
`event_cols[s]` are the ones with `step == s`, and `target` holds the true
value (with the boundary sentinel `-2` remapped to `106` by `03_splits`, undone
below).

In [1]:
import json

import numpy as np
import pandas as pd

train = pd.read_csv("data/train_set.csv")
val = pd.read_csv("data/val_set.csv")

with open("data/event_masks.json") as f:
    masks = json.load(f)
with open("data/category_lookup.json") as f:
    lookup = {col: {int(k): v for k, v in m.items()} for col, m in json.load(f).items()}

event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]
categorical = ["is_home", "event", "result"]

In [2]:
def field_targets(df, field):
    """True values of `field` across the split — the `target` column on the
    rows where that field is the one being predicted. `03_splits` stored the
    boundary sentinel as 106; put it back to -2 so numeric error on boundary
    rows is |pred - (-2)|, matching the v5 evaluation."""
    s = event_cols.index(field)
    y = df.loc[df["step"] == s, "target"].to_numpy()
    return np.where(y == 106, -2, y)

## Point predictions: mode (categorical) / median (numeric)

In [3]:
def naive_point(field):
    tr = field_targets(train, field)
    y = field_targets(val, field)
    if field in categorical:
        vals, counts = np.unique(tr, return_counts=True)
        mode = int(vals[counts.argmax()])
        return {"field": field, "predict": f"{mode} = {lookup[field].get(mode, mode)}",
                "metric": "acc", "val": (y == mode).mean()}
    med = float(np.median(tr))
    return {"field": field, "predict": f"{med:g}",
            "metric": "mae", "val": np.abs(y - med).mean()}


point = pd.DataFrame(naive_point(f) for f in event_cols)
point

,field,predict,metric,val
0,is_home,1 = home,acc,0.517566
1,event,1 = pass,acc,0.430463
2,start_x,54,mae,20.634029
3,start_y,34,mae,17.541955
4,time_elapsed,1,mae,2.192153
5,duration,1,mae,0.899186
6,result,0 = success,acc,0.826649


## Distributional: cross-entropy of the training marginal

For each field, the "prediction" is the full training-set class distribution
(Laplace-smoothed so classes unseen in train still get a finite log-prob).
The per-field cross-entropies sum to the quantity the disjoint-heads model
reports as its loss.

In [4]:
def marginal_ce(field):
    classes = list(masks[field])                    # legal values, boundary as -2
    pos = {c: i for i, c in enumerate(classes)}

    counts = np.bincount([pos[v] for v in field_targets(train, field)], minlength=len(classes)).astype(float)
    p = (counts + 1) / (counts + 1).sum()

    y = field_targets(val, field)
    return {"field": field, "n_classes": len(classes),
            "val_ce": float(np.mean([-np.log(p[pos[v]]) for v in y]))}


ce = pd.DataFrame(marginal_ce(f) for f in event_cols)
ce.loc["total", "val_ce"] = ce["val_ce"].sum()
ce

,field,n_classes,val_ce
0,is_home,3.0,0.702846
1,event,24.0,1.559665
2,start_x,107.0,4.568005
3,start_y,70.0,4.225873
4,time_elapsed,106.0,1.881414
5,duration,37.0,1.536113
6,result,7.0,0.487701
total,NaN,NaN,14.961616


## For reference: naive baseline vs. disjoint-heads model (v5), val

v5 column from `evaluations/mlp_v5_disjoint.md` (kept in `scratch/`), measured
the same way — every `step == s` row, no clean-event filter, teacher-forced.

| field | metric | naive | v5 |
|---|---|---|---|
| is_home | acc | 51.8% | 95.1% |
| event | acc | 43.0% | 75.5% |
| result | acc | 82.7% | 88.4% |
| start_x | MAE | 20.63 | 6.50 |
| start_y | MAE | 17.54 | 6.89 |
| time_elapsed | MAE | 2.19 | 0.91 |
| duration | MAE | 0.90 | 0.75 |
| **all 7** | summed CE | **14.96** | **9.573** |

`result` accuracy is the one field where the naive baseline is already
competitive — "always `success`" is right 82.7% of the time because the class
is that imbalanced (same observation as the v1 evaluation). Everywhere else the
model is well clear of the floor; the summed cross-entropy gap (~5.4 nats) is
the compact version of that.